In [ ]:
import sys
from awsglue.transforms import *
from awsglue.utils import getResolvedOptions
from pyspark.context import SparkContext
from awsglue.context import GlueContext
from awsglue.job import Job

# args = getResolvedOptions(sys.argv, ['JOB_NAME'])
sc = SparkContext()
glueContext = GlueContext(sc)
spark = glueContext.spark_session
# job = Job(glueContext)
# job.init(args['JOB_NAME'], args)

/home/glue_user/spark/python/pyspark/sql/context.py:112: FutureWarning: Deprecated in 3.0.0. Use SparkSession.builder.getOrCreate() instead.
  warnings.warn(


In [ ]:

# Script generated for node Plans
plans_dyf = glueContext.create_dynamic_frame.from_options(
    format_options={"quoteChar": "\"", "withHeader": True, "separator": ",", "optimizePerformance": False},
    connection_type="s3", 
    format="csv", 
    connection_options={"paths": ["s3://aws-bigdata-blog/artifacts/telco-data-monetization-configs-us-east-1/data/tariff_plan_desc_20190501.csv.gz"], "recurse": True}, 
    transformation_ctx="plans_dyf")

# Script generated for node Plan assignment
plan_assignment_dyf = glueContext.create_dynamic_frame.from_options(format_options={"quoteChar": "\"", "withHeader": True, "separator": ",", "optimizePerformance": False}, connection_type="s3", format="csv", connection_options={"paths": ["s3://aws-bigdata-blog/artifacts/telco-data-monetization-configs-us-east-1/data/customer_subscription_map_20190501.csv.gz"], "recurse": True}, transformation_ctx="plan_assignment_dyf")

# Script generated for node Subscribers
subscribers_dyf = glueContext.create_dynamic_frame.from_options(format_options={"quoteChar": "\"", "withHeader": True, "separator": ",", "optimizePerformance": False}, connection_type="s3", format="csv", connection_options={"paths": ["s3://aws-bigdata-blog/artifacts/telco-data-monetization-configs-us-east-1/data/crm_demographics_20190501.csv.gz"], "recurse": True}, transformation_ctx="subscribers_dyf")

In [ ]:

# Script generated for node Join
plan_assignment_subscribers_joined_dyf = Join.apply(
    frame1=plan_assignment_dyf,
    frame2=subscribers_dyf, 
    keys1=["msisdn"], 
    keys2=["msisdn"], 
    transformation_ctx="plan_assignment_subscribers_joined_dyf")

# Script generated for node Join
subscriber_plans_joined_dyf = Join.apply(frame1=plan_assignment_subscribers_joined_dyf, 
                                         frame2=plans_dyf, 
                                         keys1=["plan_id"], 
                                         keys2=["plan_id"], 
                                         transformation_ctx="subscriber_plans_joined_dyf")

# Script generated for node Select Fields
subscriber_plans_selected_dyf = SelectFields.apply(frame=subscriber_plans_joined_dyf, 
                                                   paths=["plan_id", "gender", "birth_date", "is_vip", "plan_desc", "plan_price"], 
                                                   transformation_ctx="subscriber_plans_selected_dyf")


In [ ]:

# Script generated for node Change Schema
subscriber_plans_mapped_dyf = ApplyMapping.apply(
    frame=subscriber_plans_selected_dyf, 
    mappings=[("plan_id", "string", "plan_id", "bigint"), 
            ("gender", "string", "gender", "string"), 
            ("birth_date", "string", "birth_date", "string"), 
            ("is_vip", "string", "is_vip", "boolean"), 
            ("plan_desc", "string", "plan_desc", "string"), 
            ("plan_price", "string", "plan_price", "int")], 
    transformation_ctx="subscriber_plans_mapped_dyf")


In [109]:
# Script generated for node Catalog
subscriber_plans_sink = glueContext.getSink(
    path="s3://aws-glue-temporary-361116840744-ap-south-1/example/subscriber_plans/", 
    connection_type="s3", 
    updateBehavior="UPDATE_IN_DATABASE", 
    partitionKeys=["plan_id"], 
    enableUpdateCatalog=True, 
    transformation_ctx="subscriber_plans_sink"
)

type(subscriber_plans_sink)


awsglue.data_sink.DataSink

In [80]:
subscriber_plans_sink.setCatalogInfo(catalogDatabase="default",catalogTableName="subscriber_plans")
subscriber_plans_sink.setFormat("glueparquet", compression="snappy")


In [ ]:
subscriber_plans_sink.writeFrame(subscriber_plans_mapped_dyf)
# job.commit()

In [89]:
# from pyspark.sql import SparkSession

# # Create Spark session
# spark = SparkSession.builder \
#     .appName("Glue_S3_Read_Write_CSV") \
#     .getOrCreate()

# spark.sparkContext.setLogLevel("ERROR")

# -------------------------
# READ FILES FROM S3
# -------------------------

Plans_df = spark.read \
    .option("header", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .csv("s3a://aws-bigdata-blog/artifacts/telco-data-monetization-configs-us-east-1/data/tariff_plan_desc_20190501.csv.gz")

Plans_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("s3a://glue-spark-etl-example-source/raw/plans/")


26/04/26 12:35:43 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/26 12:35:43 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


In [82]:

Planassignment_df = spark.read \
    .option("header", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .csv("s3a://aws-bigdata-blog/artifacts/telco-data-monetization-configs-us-east-1/data/customer_subscription_map_20190501.csv.gz")

Planassignment_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("s3a://glue-spark-etl-example-source/raw/plan_assignment/")


26/04/26 12:32:04 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/26 12:32:04 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


In [83]:

Subscribers_df = spark.read \
    .option("header", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .csv("s3a://aws-bigdata-blog/artifacts/telco-data-monetization-configs-us-east-1/data/crm_demographics_20190501.csv.gz")

Subscribers_df.write \
    .mode("overwrite") \
    .option("header", "true") \
    .csv("s3a://glue-spark-etl-example-source/raw/subscribers/")

26/04/26 12:32:22 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.
26/04/26 12:32:22 WARN AbstractS3ACommitterFactory: Using standard FileOutputCommitter to commit work. This is slow and potentially unsafe.


In [95]:
Plans_df = spark.read \
    .option("header", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .csv("s3a://glue-spark-etl-example-source/raw/plans")
Plans_df.printSchema()
Plans_df.show(5)


root
 |-- plan_id: string (nullable = true)
 |-- plan_desc: string (nullable = true)
 |-- plan_price: string (nullable = true)

+------------+--------------------+----------+
|     plan_id|           plan_desc|plan_price|
+------------+--------------------+----------+
|166897524362| $118 - 849 min+3 GB|       118|
|207647799828|$499 - 2980 min+7...|       499|
|179559938124|$24 - Unlimited Data|        24|
|266599625835|         $7 - 699 MB|         7|
|948388125393|       $0 - 1888 min|         0|
+------------+--------------------+----------+
only showing top 5 rows



In [96]:
Planassignment_df = spark.read \
    .option("header", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .csv("s3a://glue-spark-etl-example-source/raw/plan_assignment")

Planassignment_df.printSchema()
Planassignment_df.show(5)


root
 |-- msisdn: string (nullable = true)
 |-- plan_id: string (nullable = true)

+--------------------+-------------+
|              msisdn|      plan_id|
+--------------------+-------------+
|0EE9CF575F6DFC439...| 422426142648|
|E2F7FFF946A31AB8C...| 968763331038|
|0C59DE5F33D3F591D...| 655023658812|
|8F5590688CEEAE729...| 826022912009|
|95662E66C0FCFA137...|1056269587672|
+--------------------+-------------+
only showing top 5 rows



In [98]:
Subscribers_df = spark.read \
    .option("header", "true") \
    .option("sep", ",") \
    .option("quote", '"') \
    .csv("s3a://glue-spark-etl-example-source/raw/subscribers")

Subscribers_df.printSchema()
Subscribers_df.show(5,truncate=False)

root
 |-- msisdn: string (nullable = true)
 |-- gender: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- is_vip: string (nullable = true)

+--------------------------------+------+------------------+------+
|msisdn                          |gender|birth_date        |is_vip|
+--------------------------------+------+------------------+------+
|EA5BC180F5C0E2947574B25CE606C881|F     |16-FEB-89 00:00:00|0     |
|27F376DE3C069F239419C581D7B66A71|F     |23-NOV-89 00:00:00|0     |
|7D11DAC9BE94A7967A105650D19995D0|F     |22-JAN-75 00:00:00|0     |
|32FA7B0E9A16AC956E4F8A5F77358EB0|F     |05-OCT-88 00:00:00|1     |
|5CB1E8F587F026541885580853362236|M     |25-APR-82 00:00:00|0     |
+--------------------------------+------+------------------+------+
only showing top 5 rows



In [101]:
dfp=spark.read.parquet("s3a://glue-spark-etl-example-source/run-1777206061589-part-block-0-0-r-00002-snappy.parquet")
dfp.printSchema()
dfp.show(truncate=False)

root
 |-- gender: string (nullable = true)
 |-- birth_date: string (nullable = true)
 |-- is_vip: boolean (nullable = true)
 |-- plan_desc: string (nullable = true)
 |-- plan_price: integer (nullable = true)

+------+------------------+------+-----------------------+----------+
|gender|birth_date        |is_vip|plan_desc              |plan_price|
+------+------------------+------+-----------------------+----------+
|F     |03-MAR-68 00:00:00|false |$1277 - 2026 min+310 MB|1277      |
|M     |26-NOV-85 00:00:00|true  |$1277 - 2026 min+310 MB|1277      |
|M     |27-MAR-81 00:00:00|false |$1277 - 2026 min+310 MB|1277      |
|M     |23-SEP-74 00:00:00|false |$1277 - 2026 min+310 MB|1277      |
|F     |28-FEB-84 00:00:00|false |$1277 - 2026 min+310 MB|1277      |
|F     |02-NOV-52 00:00:00|false |$1277 - 2026 min+310 MB|1277      |
|M     |12-JAN-81 00:00:00|false |$1277 - 2026 min+310 MB|1277      |
|M     |14-JUN-73 00:00:00|false |$1277 - 2026 min+310 MB|1277      |
|M     |03-FEB-88 00: